In [1]:
from IPython.display import Markdown, display

display(Markdown("""
**Ноутбук 2: 02_Preprocessing_and_FeatureEngineering.ipynb**

Во втором ноутбуке выполнена полная предобработка данных и инжиниринг признаков. Были удалены идентификатор SK_ID_CURR, все столбцы с пропусками более 50% и все FLAG_DOCUMENT_*. 

Аномальное значение в DAYS_EMPLOYED заменено на NaN, затем создан бинарный признак EMPLOYED, а пропуски заполнены медианой по работающим. 

Добавлены новые признаки: AGE (возраст в годах), WORK_EXPERIENCE (стаж в годах), CREDIT_TO_INCOME, ANNUITY_TO_INCOME и EXT_SOURCE_MEAN (среднее двух внешних источников). 

Для числовых признаков пропуски заполнены медианой, для категориальных – константой 'Unknown'. Категориальные признаки закодированы двумя способами: LabelEncoder (для деревьев) и OneHotEncoding с масштабированием (для линейных моделей). 

Данные разделены на train/validation/test (70/15/15) со стратификацией по TARGET. 

Создан ColumnTransformer для воспроизводимой предобработки, который сохранён в artifacts.
"""))


**Ноутбук 2: 02_Preprocessing_and_FeatureEngineering.ipynb**

Во втором ноутбуке выполнена полная предобработка данных и инжиниринг признаков. Были удалены идентификатор SK_ID_CURR, все столбцы с пропусками более 50% и все FLAG_DOCUMENT_*. 

Аномальное значение в DAYS_EMPLOYED заменено на NaN, затем создан бинарный признак EMPLOYED, а пропуски заполнены медианой по работающим. 

Добавлены новые признаки: AGE (возраст в годах), WORK_EXPERIENCE (стаж в годах), CREDIT_TO_INCOME, ANNUITY_TO_INCOME и EXT_SOURCE_MEAN (среднее двух внешних источников). 

Для числовых признаков пропуски заполнены медианой, для категориальных – константой 'Unknown'. Категориальные признаки закодированы двумя способами: LabelEncoder (для деревьев) и OneHotEncoding с масштабированием (для линейных моделей). 

Данные разделены на train/validation/test (70/15/15) со стратификацией по TARGET. 

Создан ColumnTransformer для воспроизводимой предобработки, который сохранён в artifacts.


Импорт

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

Загрузка данных

In [2]:
df = pd.read_csv("../data/raw/application_train.csv")
print(f"Размер исходного датафрейма: {df.shape}")
df.head()

Размер исходного датафрейма: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Удаление признаков

In [3]:
df.drop('SK_ID_CURR', axis=1, inplace=True)

missing_percent = df.isnull().mean() * 100
high_missing_cols = missing_percent[missing_percent > 50].index.tolist()
print(f"Удаляем {len(high_missing_cols)} признаков с пропусками >50%")
df.drop(columns=high_missing_cols, inplace=True, errors='ignore')

flag_cols = [col for col in df.columns if col.startswith('FLAG_DOCUMENT_')]
df.drop(columns=flag_cols, inplace=True, errors='ignore')
print(f"Итоговое количество признаков: {df.shape[1]}")

Удаляем 41 признаков с пропусками >50%
Итоговое количество признаков: 60


Обработка выбросов в DAYS_EMPLOYED

In [4]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

df['EMPLOYED'] = df['DAYS_EMPLOYED'].notna().astype(int)

median_employed = df[df['EMPLOYED'] == 1]['DAYS_EMPLOYED'].median()
df['DAYS_EMPLOYED'].fillna(median_employed, inplace=True)

print(f"Медиана DAYS_EMPLOYED для работающих: {median_employed}")
print(f"Пропусков после обработки: {df['DAYS_EMPLOYED'].isnull().sum()}")

Медиана DAYS_EMPLOYED для работающих: -1648.0
Пропусков после обработки: 0


C:\Users\varva\AppData\Local\Temp\ipykernel_14616\2805001209.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['DAYS_EMPLOYED'].fillna(median_employed, inplace=True)


Создание новых признаков

In [5]:
# Возраст (положительный)
df['AGE'] = -df['DAYS_BIRTH'] // 365

# Стаж работы (положительный)
df['WORK_EXPERIENCE'] = -df['DAYS_EMPLOYED'] // 365

# Отношение суммы кредита к доходу
df['CREDIT_TO_INCOME'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1e-5)

# Отношение аннуитета к доходу
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1e-5)

# Среднее по внешним источникам (если есть пропуски, заполним медианой)
ext_cols = ['EXT_SOURCE_2', 'EXT_SOURCE_3']
for col in ext_cols:
    df[col].fillna(df[col].median(), inplace=True)
df['EXT_SOURCE_MEAN'] = df[ext_cols].mean(axis=1)

print("Новые признаки добавлены")

Новые признаки добавлены


C:\Users\varva\AppData\Local\Temp\ipykernel_14616\2698551554.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\varva\AppData\Local\Temp\ipykernel_14616\2698551554.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

Заполнение пропусков в остальных признаках

In [6]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'TARGET' in numeric_cols:
    numeric_cols.remove('TARGET')
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Числовых признаков: {len(numeric_cols)}")
print(f"Категориальных признаков: {len(categorical_cols)}")

Числовых признаков: 52
Категориальных признаков: 13


In [7]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

In [8]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print(f"Осталось пропусков: {df.isnull().sum().sum()}")

Осталось пропусков: 0


Кодирование категориальных признаков

Для деревьев (LightGBM) буду использовать LabelEncoder.
Для линейных моделей и MLP OneHotEncoding.
Так как я буду сравнивать несколько моделей, создадим две версии данных.

LabelEncoder для категориальных признаков

In [9]:
from sklearn.preprocessing import LabelEncoder

df_label = df.copy()
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_label[col] = le.fit_transform(df_label[col].astype(str))
    le_dict[col] = le

X_tree = df_label.drop('TARGET', axis=1)
y = df_label['TARGET']

print(f"Форма X для деревьев: {X_tree.shape}")

Форма X для деревьев: (307511, 65)


Разделение на train / validation / test

In [10]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_tree, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Доля дефолтов в train: {y_train.mean():.2%}")
print(f"Доля дефолтов в val: {y_val.mean():.2%}")
print(f"Доля дефолтов в test: {y_test.mean():.2%}")

Train: (215249, 65), Val: (46135, 65), Test: (46127, 65)
Доля дефолтов в train: 8.07%
Доля дефолтов в val: 8.07%
Доля дефолтов в test: 8.07%


Создание пайплайна предобработки для логистической регрессии и MLP

In [11]:
X_linear = df.drop('TARGET', axis=1)
y_linear = df['TARGET']

X_train_lin, X_temp_lin, y_train_lin, y_temp_lin = train_test_split(
    X_linear, y_linear, test_size=0.3, random_state=42, stratify=y_linear
)
X_val_lin, X_test_lin, y_val_lin, y_test_lin = train_test_split(
    X_temp_lin, y_temp_lin, test_size=0.5, random_state=42, stratify=y_temp_lin
)

print(f"Train: {X_train_lin.shape}, Val: {X_val_lin.shape}, Test: {X_test_lin.shape}")
print(f"Доля дефолтов: train={y_train_lin.mean():.2%}, val={y_val_lin.mean():.2%}, test={y_test_lin.mean():.2%}")

numeric_cols_lin = X_train_lin.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_lin = X_train_lin.select_dtypes(include=['object']).columns.tolist()

print(f"Числовых признаков: {len(numeric_cols_lin)}")
print(f"Категориальных признаков: {len(categorical_cols_lin)}")

# Пайплайн для числовых: заполнение медианой + масштабирование
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных: заполнение 'Unknown' + OneHot (drop first)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols_lin),
        ('cat', categorical_transformer, categorical_cols_lin)
    ]
)

preprocessor.fit(X_train_lin)

X_train_lin_processed = preprocessor.transform(X_train_lin)
X_val_lin_processed = preprocessor.transform(X_val_lin)
X_test_lin_processed = preprocessor.transform(X_test_lin)

print(f"Размер после трансформации: train={X_train_lin_processed.shape}, val={X_val_lin_processed.shape}, test={X_test_lin_processed.shape}")

joblib.dump(preprocessor, '../artifacts/preprocessor.pkl')
print("Препроцессор сохранён в '../artifacts/preprocessor.pkl'")

Train: (215257, 65), Val: (46127, 65), Test: (46127, 65)
Доля дефолтов: train=8.07%, val=8.07%, test=8.07%
Числовых признаков: 52
Категориальных признаков: 13
Размер после трансформации: train=(215257, 168), val=(46127, 168), test=(46127, 168)
Препроцессор сохранён в '../artifacts/preprocessor.pkl'


Сохранение артефактов

In [12]:
# Сохраняем данные для обучения (в папку data/processed)
X_train_lin.to_csv('../data/processed/X_train.csv', index=False)
y_train_lin.to_csv('../data/processed/y_train.csv', index=False)
X_val_lin.to_csv('../data/processed/X_val.csv', index=False)
y_val_lin.to_csv('../data/processed/y_val.csv', index=False)
X_test_lin.to_csv('../data/processed/X_test.csv', index=False)
y_test_lin.to_csv('../data/processed/y_test.csv', index=False)

# Для деревьев сохраняем отдельно (уже закодированные)
X_train.to_csv('../data/processed/X_train_tree.csv', index=False)
X_val.to_csv('../data/processed/X_val_tree.csv', index=False)
X_test.to_csv('../data/processed/X_test_tree.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)  # целевые общие
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Все артефакты сохранены")

Все артефакты сохранены


In [13]:
y_train_lin.to_csv('../data/processed/y_train_lin.csv', index=False)
y_val_lin.to_csv('../data/processed/y_val_lin.csv', index=False)
y_test_lin.to_csv('../data/processed/y_test_lin.csv', index=False)
print("Целевые для линейных моделей сохранены в y_train_lin.csv, y_val_lin.csv, y_test_lin.csv")

Целевые для линейных моделей сохранены в y_train_lin.csv, y_val_lin.csv, y_test_lin.csv
